In [2]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.linear_model import (
    ElasticNet,
    Lasso,
    LinearRegression,
    Ridge,
    ridge_regression,
)
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures

# Set working directory (using a raw string to avoid escape character issues with backslashes)
os.chdir(
    r"E:\Backup Files Abhinav_Satyam\Practical Machine Learning\Sanjay Sir Git Repo\Datasets\Cases\Medical_Cost_Personal"
)

In [3]:
df = pd.read_csv('insurance.csv')
df

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [6]:
x,y = df.drop('charges', axis=1), df['charges']

X_train, X_test, y_train, y_test = train_test_split(x,y,random_state=26, test_size=0.3)


In [7]:
ohe = OneHotEncoder(sparse_output=False, drop='first').set_output(transform='pandas')

In [8]:
ct = make_column_transformer(
                            (ohe, make_column_selector(dtype_include = object)),
                            ('passthrough', make_column_selector(dtype_include =['int64','float64'])),
                             verbose_feature_names_out = False)
ct = ct.set_output(transform = 'pandas')
x_ohe = ct.fit_transform(X_train)
x_ohe

,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest,age,bmi,children
219,0.0,0.0,0.0,1.0,0.0,24,23.21,0
351,0.0,0.0,0.0,0.0,1.0,50,25.60,0
824,1.0,0.0,1.0,0.0,0.0,60,24.32,0
484,1.0,0.0,0.0,0.0,1.0,48,34.30,3
647,0.0,0.0,0.0,0.0,0.0,40,23.37,3
...,...,...,...,...,...,...,...,...
941,0.0,0.0,0.0,1.0,0.0,50,46.09,1
160,0.0,1.0,1.0,0.0,0.0,42,26.60,0
1101,1.0,0.0,0.0,0.0,1.0,53,28.60,3
1126,1.0,0.0,0.0,0.0,1.0,55,29.90,0


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](8,)","[ -106.42,23682.21, -127.98,..., 240.51, 362.48, 292.22]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](8,)","['sex_male','smoker_yes','region_northwest',...,'age','bmi','children']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-1.183e+04
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,8
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(8)


In [22]:
# One-hot-encoding on test data
X_test_transf = ct.transform(X_test)

In [14]:
lr = LinearRegression()
lr.fit(x_ohe, y_train)

y_pred = lr.predict(X_test_transf)
r2score = r2_score(y_test,y_pred)
r2score

0.7625873682287879

In [19]:
alphas = np.linspace(0.001, 10.0,20) 

lst = []

for alp in alphas:
    ridg = Ridge(alpha=alp)
    ridg.fit(x_ohe, y_train)
    y_pred = ridg.predict(X_test_transf)
    lst.append([alp,r2_score(y_test, y_pred)]) 

valsdic = pd.DataFrame(lst, columns = ['alpha','ridge r2 score'])
valsdic.sort_values('ridge r2 score',ascending = False)

,alpha,ridge r2 score
0,0.001000,0.762587
1,0.527263,0.762494
2,1.053526,0.762387
3,1.579789,0.762265
4,2.106053,0.762131
5,2.632316,0.761983
6,3.158579,0.761822
7,3.684842,0.761649
8,4.211105,0.761463
9,4.737368,0.761266


In [20]:
valsdic = pd.DataFrame(index = alphas)

lst = []
for alp in alphas:
    lasso = Lasso(alpha=alp)
    lasso.fit(x_ohe, y_train)
    y_pred = lasso.predict(X_test_transf)
    lst.append([alp,r2_score(y_test, y_pred)]) 

valsdic = pd.DataFrame(lst,columns = ['alpha','lasso r2 score'])
valsdic.sort_values('lasso r2 score',ascending = False)

,alpha,lasso r2 score
0,0.001000,0.762587
1,0.527263,0.762585
2,1.053526,0.762583
3,1.579789,0.762580
4,2.106053,0.762577
5,2.632316,0.762574
6,3.158579,0.762570
7,3.684842,0.762566
8,4.211105,0.762563
9,4.737368,0.762558


In [21]:

l1_ratios = np.linspace(0.001,1,20)
scores = []
for alp in alphas:
    for r in l1_ratios:
        elnet = ElasticNet(alpha=alp, l1_ratio = r)
        elnet.fit(x_ohe, y_train)
        y_pred = elnet.predict(X_test_transf)
        scores.append([alp,r,r2_score(y_test, y_pred)])

 

# 2. Convert the list of results into a DataFrame
df_scores = pd.DataFrame(scores, columns=['alpha', 'l1_ratio', 'r2_score'])
df_scores.sort_values('r2_score',ascending = False)
 

,alpha,l1_ratio,r2_score
19,0.001000,1.000000,0.762587
39,0.527263,1.000000,0.762585
59,1.053526,1.000000,0.762583
79,1.579789,1.000000,0.762580
18,0.001000,0.947421,0.762579
...,...,...,...
382,10.000000,0.106158,0.164873
361,9.473737,0.053579,0.164799
360,9.473737,0.001000,0.163410
381,10.000000,0.053579,0.163404


In [23]:
# find out best model -> lr

#inferencing 
''' 
1. fit the best model on whole data
2. Apply it on unlabelled data
'''

' \n1. fit the best model on whole data\n2. Apply it on unlabelled data\n'

In [24]:
df = pd.read_csv('insurance.csv')
df

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [26]:
x,y = df.drop(columns = ['charges']), df['charges']
 
ohe = OneHotEncoder(sparse_output=False, drop='first').set_output(transform='pandas')
ct = make_column_transformer(
                            (ohe, make_column_selector(dtype_include = object)),
                            ('passthrough', make_column_selector(dtype_include =['int64','float64'])),
                             verbose_feature_names_out = False)
ct = ct.set_output(transform = 'pandas')
x_ohe = ct.fit_transform(x)
x_ohe

,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest,age,bmi,children
0,0.0,1.0,0.0,0.0,1.0,19,27.900,0
1,1.0,0.0,0.0,1.0,0.0,18,33.770,1
2,1.0,0.0,0.0,1.0,0.0,28,33.000,3
3,1.0,0.0,1.0,0.0,0.0,33,22.705,0
4,1.0,0.0,1.0,0.0,0.0,32,28.880,0
...,...,...,...,...,...,...,...,...
1333,1.0,0.0,1.0,0.0,0.0,50,30.970,3
1334,0.0,0.0,0.0,0.0,0.0,18,31.920,0
1335,0.0,0.0,0.0,1.0,0.0,18,36.850,0
1336,0.0,0.0,0.0,0.0,1.0,21,25.800,0


In [31]:
lr = LinearRegression()
lr.fit(x_ohe, y)

testdf = pd.read_csv('tst_insure.csv')
testenc = ct.transform(testdf)

pred = lr.predict(testenc)  
pred

print('predictions => ')
testdf['predicted charges'] = pred
testdf

predictions => 


,age,sex,bmi,children,smoker,region,predicted charges
0,19,female,27.900,0,yes,southwest,25293.713028
1,18,male,33.770,1,no,southeast,3448.602834
2,28,male,33.000,3,no,southeast,6706.988491
3,33,male,22.705,0,no,northwest,3754.830163
4,37,female,23.370,2,no,northwest,6090.134669
...,...,...,...,...,...,...,...
83,60,male,24.320,1,no,northwest,11713.249654
84,51,female,36.670,2,no,northwest,14197.396538
85,58,female,33.440,0,no,northwest,13948.795060
86,36,male,34.430,2,no,southeast,8771.385405
